# Evaluation of all Models for Species Classifcation

In this notebook I will evaluate all models and explain all approaches I tried in order to increase model accuracy.

In [1]:
from species_classification import load_model, create_binary_dataset, train_test_split
import matplotlib.pyplot as plt
import numpy as np

# loading datasets
train = np.load("../train.npz")
X_train = train["X"]
y_train = train["y"]

val = np.load("../val.npz")
X_val = val["X"]
y_val = val["y"]

test = np.load("../test.npz")
X_test = test["X"]
y_test = test["y"]

X_train_rgb = X_train[:, :, :, 1:]
X_val_rgb = X_val[:, :, :, 1:]
X_test_rgb = X_test[:, :, :, 1:]

X_train_thermal = X_train[:, :, :, 0:1]
X_val_thermal = X_val[:, :, :, 0:1]
X_test_thermal = X_test[:, :, :, 0:1]

I0000 00:00:1781879482.239331   17639 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Data Preprocessing

I cropped the original thermal and RGB images based on the corresponding YOLO annotations. To preserve the image format and to ensure the images are all of the same size, I padded the images to a shape of 128x128. 

The RGB images were normalized by dividing by 255. For the thermal images I did z-score normalization as this makes more sense for images of these kind. 

## Custom CNN
My first approach was to train a small custom CNN on the thermal images. Unfortunately, it did not lead to good results. 

In [2]:
load_model("../species_class_models/thermal_basic.keras", X_train_thermal, y_train, X_val_thermal, y_val, X_test_thermal, y_test)

I0000 00:00:1781855641.093277   14287 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14793 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0001:00:00.0, compute capability: 7.5


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ sequential (Sequential)              │ (None, 128, 128, 1)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 126, 126, 32)        │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 126, 126, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 63, 63, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 61, 61, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 61, 61, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 30, 30, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 28, 28, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 28, 28, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 14, 14, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 12, 12, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 12, 12, 256)         │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 256)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 8)                   │           2,056 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 8)                   │              32 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 8)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 5)                   │              45 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,173,729 (4.48 MB)

 Trainable params: 390,917 (1.49 MB)

...

None


W0000 00:00:1781855644.836993   14287 cpu_allocator_impl.cc:82] Allocation of 2033909760 exceeds 10% of free system memory.
W0000 00:00:1781855645.306969   14287 cpu_allocator_impl.cc:82] Allocation of 2033909760 exceeds 10% of free system memory.
I0000 00:00:1781855646.287459   16077 cuda_dnn.cc:461] Loaded cuDNN version 91002


970/970 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.7679 - loss: 0.5584
Train loss: 0.5584203600883484
Train accuracy: 0.7679072022438049


W0000 00:00:1781855658.868253   14287 cpu_allocator_impl.cc:82] Allocation of 2033909760 exceeds 10% of free system memory.
W0000 00:00:1781855659.336042   14287 cpu_allocator_impl.cc:82] Allocation of 2033909760 exceeds 10% of free system memory.


970/970 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step
[[1953  462   37  295    4]
 [ 274 8855   58  861    0]
 [   3   53 3970   14    0]
 [ 678 3805  170 8770   71]
 [ 149   23    2  244  284]]
Balanced accuracy: 0.7256687445508596
              precision    recall  f1-score   support

           0      0.639     0.710     0.673      2751
           1      0.671     0.881     0.762     10048
           2      0.937     0.983     0.959      4040
           3      0.861     0.650     0.741     13494
           4      0.791     0.405     0.535       702

    accuracy                          0.768     31035
   macro avg      0.780     0.726     0.734     31035
weighted avg      0.788     0.768     0.765     31035

170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.4721 - loss: 1.8818
Val loss: 1.8818249702453613
Val accuracy: 0.4721093475818634
170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
[[ 221 1112   50  349    0]
 [ 402 1801   32  263   10]
 [  48  137    0   29    0]
 [  38  143   66  534    0]
 

While train accuarcy is above 70%, validation is at 0.47 (balanced only 0.3), and test at 0.42 (with balanced accuracy of 0.27 barely better than random). The two most difficult classes to predict are Fallow Deer (class 2) and Hybrid Pig (class 4) as these are the classes with the least samples in the validation and test dataset. Interestingly, no Hybrid Pigs in the validation dataset were correctly predicted, while 20 in the test dataset. For both, no class 2 was correctly predicted. In case of the test dataset, all 12 samples were classified as Red Deer. 

I tried improving the model performance by applying Data Augmentation, Oversampling, Balanced Class Weights that punish misclassifications of smaller classes more heavily. All of them did either not affect model performance or, in some cases, even made it worse. 

Data Augmentation in some cases confuses the model more that it helps it learning the actual animal classes. I tried different augmentations that are provided by keras and discovered that less or even no augmentation layers result in better model performance. Oversampling did not affect the resulting accuracies much, and the balanced class weights just led to very high loss values because the species are very unevenly distributed. (For a closer look at the distributions, see `eda_species_classification.ipynb`) 

Additionally, I tried many different model architectures and hyperparameter combinations that all did not improve the model performance. 

## Custom CNN on Thermal & RGB Images

As training on thermal images only did not result in high accuracy scores, I also downloaded the corresponding RGB images for this dataset. 
Adding the RGB images to the input increased the test accuracy immediately, but unfortunately not much. A possible reason for the only slight improvement is that many labels for the RGB images are misaligned, leading to the animal being not in the center of the cropped image and in some cases even being cut off.

In [2]:
load_model("../species_class_models/rgb_normal.keras", X_train, y_train, X_val, y_val, X_test, y_test)

I0000 00:00:1781802151.264540   44158 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14793 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0001:00:00.0, compute capability: 7.5


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 126, 126, 32)        │           1,184 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 126, 126, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 63, 63, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 61, 61, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 61, 61, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 30, 30, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 28, 28, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 28, 28, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 14, 14, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 12, 12, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 12, 12, 256)         │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 256)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 8)                   │           2,056 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 8)                   │              32 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 8)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 5)                   │              45 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,176,321 (4.49 MB)

 Trainable params: 391,781 (1.49 MB)

...

970/970 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.8840 - loss: 0.3253
Train loss: 0.32532402873039246
Train accuracy: 0.8840019106864929


W0000 00:00:1781802191.506184   44158 cpu_allocator_impl.cc:82] Allocation of 8135639040 exceeds 10% of free system memory.
W0000 00:00:1781802206.980045   44158 cpu_allocator_impl.cc:82] Allocation of 8135639040 exceeds 10% of free system memory.


970/970 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step
[[ 2331   262     4   153     1]
 [   68  9007    13   960     0]
 [  114    23  3820    80     3]
 [  121  1223    13 12119    18]
 [    5    22     0   517   158]]
Balanced accuracy: 0.7624888354651432
              precision    recall  f1-score   support

           0      0.883     0.847     0.865      2751
           1      0.855     0.896     0.875     10048
           2      0.992     0.946     0.968      4040
           3      0.876     0.898     0.887     13494
           4      0.878     0.225     0.358       702

    accuracy                          0.884     31035
   macro avg      0.897     0.762     0.791     31035
weighted avg      0.885     0.884     0.880     31035



W0000 00:00:1781802220.715488   44158 cpu_allocator_impl.cc:82] Allocation of 1419247616 exceeds 10% of free system memory.


170/170 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.6284 - loss: 1.5204
Val loss: 1.5203745365142822
Val accuracy: 0.6283708810806274
170/170 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step
[[1572   95   39   26    0]
 [ 521 1138    3  843    3]
 [  61   68    0   85    0]
 [   1  125    0  636   19]
 [   0    1    0  122   56]]
Balanced accuracy: 0.4977118008984798
              precision    recall  f1-score   support

           0      0.729     0.908     0.809      1732
           1      0.797     0.454     0.578      2508
           2      0.000     0.000     0.000       214
           3      0.371     0.814     0.510       781
           4      0.718     0.313     0.436       179

    accuracy                          0.628      5414
   macro avg      0.523     0.498     0.467      5414
weighted avg      0.680     0.628     0.615      5414

145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.6238 - loss: 1.3629
Test loss: 1.3628870248794556
Test accuracy: 0.6238116025924683
145/145 ━━━━━

Validation and Test accuracy are around 60%, balanced accuracy only merely over 40% for the test dataset. While this is an improvement compared to training on the thermal data only, these accuracy scores are still not very good. The biggest problem is that not a single sample of class 2 (Fallow Deer) has been classified correctly in the validation and test dataset, while this class has the highest precision score of all classes on the training dataset. For the second smallest class in the test dataset, Hybrid pig, at least 35 samples were correctly predicted. 

## Transfer Learning 

At first I was doing transfer learning with the EfficientNetV2B0 on the thermal images. Since EfficientNet was trained on RGB images, it was not really better at classifying the animals than my custom CNN. 

As a result, I tried transfer learning on the RGB images. The problems here are that often the animals are hidden behind trees or the RGB labels are not correct, in the worst case there might not even be an animal or only a part of it in the cropped image. 

In [3]:
load_model("../species_class_models/transfer.keras", X_train_rgb, y_train, X_val_rgb, y_val, X_test_rgb, y_test)

I0000 00:00:1781878657.953567    6392 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14793 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0001:00:00.0, compute capability: 7.5


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 128, 128, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ sequential (Sequential)              │ (None, 128, 128, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ efficientnetv2-b0 (Functional)       │ (None, 4, 4, 1280)          │       5,919,312 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 512)                 │         655,872 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 5)                   │           1,285 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 8,284,769 (31.60 MB)

 Trainable params: 788,485 (3.01 MB)

...

/home/azureuser/Projects/CVI_Project/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/azureuser/Projects/CVI_Project/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/azureuser/Projects/CVI_Project/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

170/170 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.1443 - loss: 1.5771
Val loss: 1.5771242380142212
Val accuracy: 0.1442556381225586
170/170 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step
[[   0    0    0 1732    0]
 [   0    0    0 2508    0]
 [   0    0    0  214    0]
 [   0    0    0  781    0]
 [   0    0    0  179    0]]
Balanced accuracy: 0.2
              precision    recall  f1-score   support

           0      0.000     0.000     0.000      1732
           1      0.000     0.000     0.000      2508
           2      0.000     0.000     0.000       214
           3      0.144     1.000     0.252       781
           4      0.000     0.000     0.000       179

    accuracy                          0.144      5414
   macro avg      0.029     0.200     0.050      5414
weighted avg      0.021     0.144     0.036      5414



/home/azureuser/Projects/CVI_Project/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/azureuser/Projects/CVI_Project/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/azureuser/Projects/CVI_Project/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - accuracy: 0.4198 - loss: 1.3706
Test loss: 1.3706377744674683
Test accuracy: 0.41983577609062195
145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step
[[   0    0    0  957    0]
 [   0    0    0 1552    0]
 [   0    0    0   12    0]
 [   0    0    0 1943    0]
 [   0    0    0  164    0]]
Balanced accuracy: 0.2
              precision    recall  f1-score   support

           0      0.000     0.000     0.000       957
           1      0.000     0.000     0.000      1552
           2      0.000     0.000     0.000        12
           3      0.420     1.000     0.591      1943
           4      0.000     0.000     0.000       164

    accuracy                          0.420      4628
   macro avg      0.084     0.200     0.118      4628
weighted avg      0.176     0.420     0.248      4628



/home/azureuser/Projects/CVI_Project/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/azureuser/Projects/CVI_Project/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/azureuser/Projects/CVI_Project/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

Looking at the results, unfortunately the model failed to actually learn what the animals look like just classifies everything as Wild Boar, which is the class in the training dataset with the most samples.

To investigate whether the animals looking too similar to each other causes the low model accuracies or the flights of the different splits being too different, I tried binary classificaiton and a custom train-, validation- and test-split. 

## Custom Split
I suspect that the models struggles with classifying images of flights it has never seen before or overall with the very different species distributions in the different datasets. Therefore, I combined the original train, validation, and test datasets and split them customly. I also set the stratify parameter to ensure the same distribution in every dataset.

I also applied Data Augmentation: 
```python
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])
```

In [9]:
X = np.concatenate([X_train_thermal, X_val_thermal, X_test_thermal])
y = np.concatenate([y_train, y_val, y_test])

X_train1, X_temp, y_train1, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_val1, X_test1, y_val1, y_test1 = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

load_model("../species_class_models/custom_split.keras", X_train1, y_train1, X_val1, y_val1, X_test1, y_test1)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ sequential (Sequential)              │ (None, 128, 128, 1)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 126, 126, 64)        │             640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 126, 126, 64)        │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 63, 63, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 61, 61, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 61, 61, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 30, 30, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 28, 28, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 28, 28, 256)         │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 256)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │           8,224 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 32)                  │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 5)                   │             165 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,138,001 (4.34 MB)

 Trainable params: 379,013 (1.45 MB)

 Non-trainable params: 960 (3.75 KB)

 Optimizer params: 758,028 (2.89 MB)

None
1027/1027 ━━━━━━━━━━━━━━━━━━━━ 25s 12ms/step - accuracy: 0.7702 - loss: 0.5767
Train loss: 0.5767130255699158
Train accuracy: 0.7702139019966125
1027/1027 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step
[[ 3259   569    16   503     5]
 [  817  8317   133  2018     1]
 [   42    59  3278    34     0]
 [  981  1619   187 10101    86]
 [  142    13     3   323   355]]
Balanced accuracy: 0.7298850864203419
              precision    recall  f1-score   support

           0      0.622     0.749     0.679      4352
           1      0.786     0.737     0.761     11286
           2      0.906     0.960     0.933      3413
           3      0.778     0.779     0.778     12974
           4      0.794     0.425     0.553       836

    accuracy                          0.770     32861
   macro avg      0.777     0.730     0.741     32861
weighted avg      0.774     0.770     0.770     32861

129/129 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.7610 - loss: 0.6059
Val loss: 0.605883002281189
Val accu

With my custom split, I reach train, validation, and test accuracies above 70% without much hyperparameter finetuning. This indicates to me that the differences between the flights are the real problem and reason why the other models fail to reach high accuracy scores. 

## Binary Classification 
I tried binary classification between deers and pigs. Red Deer, Roe Deer, and Fallow Deer make up class 0, class 1 consists of Wild Boar and Hybrid Pig. 

I also applied Data Augmentation: 
```python
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1)
])
```

In [5]:
y_train_bin = create_binary_dataset(y_train)
y_val_bin = create_binary_dataset(y_val)
y_test_bin = create_binary_dataset(y_test)

### Only Thermal Images

In [5]:
load_model("../species_class_models/binary_thermal.keras", X_train_thermal, y_train_bin, X_val_thermal, y_val_bin, X_test_thermal, y_test_bin)

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ sequential_2 (Sequential)            │ (None, 128, 128, 1)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 126, 126, 64)        │             640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 126, 126, 64)        │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 63, 63, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 61, 61, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_5                │ (None, 61, 61, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 30, 30, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 28, 28, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_6                │ (None, 28, 28, 256)         │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_1           │ (None, 256)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 32)                  │           8,224 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_7                │ (None, 32)                  │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 2)                   │              66 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,137,704 (4.34 MB)

 Trainable params: 378,914 (1.45 MB)

 Non-trainable params: 960 (3.75 KB)

 Optimizer params: 757,830 (2.89 MB)

None
970/970 ━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step - accuracy: 0.8096 - loss: 0.4073
Train loss: 0.4073432385921478
Train accuracy: 0.8096020817756653
970/970 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step
[[15396  1443]
 [ 4466  9730]]
Balanced accuracy: 0.7998552072165523
              precision    recall  f1-score   support

           0      0.775     0.914     0.839     16839
           1      0.871     0.685     0.767     14196

    accuracy                          0.810     31035
   macro avg      0.823     0.800     0.803     31035
weighted avg      0.819     0.810     0.806     31035

170/170 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.8519 - loss: 0.3631
Val loss: 0.3631276488304138
Val accuracy: 0.8518655300140381
170/170 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step
[[3963  491]
 [ 311  649]]
Balanced accuracy: 0.7829018391707828
              precision    recall  f1-score   support

           0      0.927     0.890     0.908      4454
           1      0.569     0.676     0.618       960

    

Without much hyperparameter finetuning, the binary classification model achieved a validation accuracy of 0.85 and a test accuracy of 0.67. These considerably high accuracies (compared to classifying 5 species) definitely indicate that the animals looking similarly makes it harder for the model to classify them correctly. However, as the test accuracy is still quite low, I assume there are also other issues contributing to the bad model performance. 

### Only RGB-Images

In [7]:
load_model("../species_class_models/binary_rgb.keras", X_train_rgb, y_train_bin, X_val_rgb, y_val_bin, X_test_rgb, y_test_bin)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ sequential (Sequential)              │ (None, 128, 128, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 126, 126, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 126, 126, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 63, 63, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 61, 61, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 61, 61, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 30, 30, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 28, 28, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 28, 28, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 14, 14, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 12, 12, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 12, 12, 256)         │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 256)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 8)                   │           2,056 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 8)                   │              32 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 8)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 2)                   │              18 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,175,376 (4.48 MB)

 Trainable params: 391,466 (1.49 MB)

 Non-trainable params: 976 (3.81 KB)

 Optimizer params: 782,934 (2.99 MB)

None
970/970 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - accuracy: 0.7882 - loss: 0.4914
Train loss: 0.4913727045059204
Train accuracy: 0.788239061832428
970/970 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step
[[12511  4328]
 [ 2244 11952]]
Balanced accuracy: 0.7924524574814424
              precision    recall  f1-score   support

           0      0.848     0.743     0.792     16839
           1      0.734     0.842     0.784     14196

    accuracy                          0.788     31035
   macro avg      0.791     0.792     0.788     31035
weighted avg      0.796     0.788     0.788     31035

170/170 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.7525 - loss: 0.5445
Val loss: 0.5444570183753967
Val accuracy: 0.7524935603141785
170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
[[3149 1305]
 [  35  925]]
Balanced accuracy: 0.8352733030234994
              precision    recall  f1-score   support

           0      0.989     0.707     0.825      4454
           1      0.415     0.964     0.580       960

    accura

Training the binary classification model only on the RGB images achieved slightly higher test accuracies. 

### Thermal & RGB Images

In [8]:
load_model("../species_class_models/binary_classification.keras", X_train, y_train_bin, X_val, y_val_bin, X_test, y_test_bin)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ sequential (Sequential)              │ (None, 128, 128, 4)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 126, 126, 64)        │           2,368 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 126, 126, 64)        │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 63, 63, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 61, 61, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 61, 61, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 30, 30, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 28, 28, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 28, 28, 256)         │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 256)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │           8,224 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 32)                  │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 2)                   │              66 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,142,888 (4.36 MB)

 Trainable params: 380,642 (1.45 MB)

 Non-trainable params: 960 (3.75 KB)

 Optimizer params: 761,286 (2.90 MB)

None
970/970 ━━━━━━━━━━━━━━━━━━━━ 14s 14ms/step - accuracy: 0.8768 - loss: 0.3168
Train loss: 0.3167765736579895
Train accuracy: 0.8768165111541748
970/970 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step
[[15535  1304]
 [ 2519 11677]]
Balanced accuracy: 0.8725581858059348
              precision    recall  f1-score   support

           0      0.860     0.923     0.890     16839
           1      0.900     0.823     0.859     14196

    accuracy                          0.877     31035
   macro avg      0.880     0.873     0.875     31035
weighted avg      0.878     0.877     0.876     31035

170/170 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.8679 - loss: 0.3839
Val loss: 0.38390302658081055
Val accuracy: 0.8679350018501282
170/170 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step
[[4006  448]
 [ 267  693]]
Balanced accuracy: 0.8106456275258196
              precision    recall  f1-score   support

           0      0.938     0.899     0.918      4454
           1      0.607     0.722     0.660       960

   

Doing binary classification with the thermal and the RGB images as input increased slightly the validation accuracy to 0.86, but unfortunately the test accuracy decreased about 10% to only 0.57.

As the models trained on the thermal and the RGB images separately performed better than training on both at the same time, I assume that simply concatenate the two kinds of images does not improve the model performance. 

## Feature-Level Fusion of RGB and Thermal Images

To exploit the complementary information contained in RGB and thermal imagery, I also tried a feature-level fusion architecture. Both image types were processed separately with a custom CNN from scratch. The feature vectors extracted by both branches were concatenated and passed to fully connected layers.

I tried processing the RGB images with EfficientNet, but using a custom CNN actually provided me with better results. 

In [2]:
load_model("../species_class_models/combined.keras", [X_train_rgb, X_train_thermal], y_train, [X_val_rgb, X_val_thermal], y_val, [X_test_rgb, X_test_thermal], y_test)

I0000 00:00:1781879521.943417   17639 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14793 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0001:00:00.0, compute capability: 7.5


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 128, 128, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_1 (InputLayer)    │ (None, 128, 128, 1)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d (Conv2D)               │ (None, 128, 128, 32)      │             896 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_3 (Conv2D)             │ (None, 128, 128, 32)      │             320 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 128, 128, 32)      │             128 │ conv2d[0][0]               │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_3         │ (None, 128, 128, 32)      │             128 │ conv2d_3[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d (MaxPooling2D)  │ (None, 64, 64, 32)        │               0 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_2               │ (None, 64, 64, 32)        │               0 │ batch_normalization_3[0][… │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_1 (Conv2D)             │ (None, 64, 64, 64)        │          18,496 │ max_pooling2d[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_4 (Conv2D)             │ (None, 64, 64, 64)        │          18,496 │ max_pooling2d_2[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 64, 64, 64)        │             256 │ conv2d_1[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_4         │ (None, 64, 64, 64)        │             256 │ conv2d_4[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_1               │ (None, 32, 32, 64)        │               0 │ batch_normalization_1[0][… │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_3               │ (None, 32, 32, 64)        │               0 │ batch_normalization_4[0][… │
│ (MaxPooling2D)                │                           │               

 Total params: 611,665 (2.33 MB)

 Trainable params: 203,589 (795.27 KB)

...

 19/970 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.7615 - loss: 0.5761 

I0000 00:00:1781879554.253171   18147 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


970/970 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - accuracy: 0.7940 - loss: 0.5499
Train loss: 0.5499075651168823
Train accuracy: 0.7940067648887634


W0000 00:00:1781879570.662226   17639 cpu_allocator_impl.cc:82] Allocation of 6101729280 exceeds 10% of free system memory.


970/970 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step
[[ 2402   122    24   150    53]
 [  497  7217   147  1836   351]
 [   11     4  4024     1     0]
 [  411  1472   288 10309  1014]
 [    1     0     4     7   690]]
Balanced accuracy: 0.8668608375922011
              precision    recall  f1-score   support

           0      0.723     0.873     0.791      2751
           1      0.819     0.718     0.765     10048
           2      0.897     0.996     0.944      4040
           3      0.838     0.764     0.799     13494
           4      0.327     0.983     0.491       702

    accuracy                          0.794     31035
   macro avg      0.721     0.867     0.758     31035
weighted avg      0.818     0.794     0.799     31035

170/170 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.6117 - loss: 1.3925
Val loss: 1.3925434350967407
Val accuracy: 0.6117473244667053
170/170 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step
[[1624    8   34   25   41]
 [ 501 1013   37  384  573]
 [ 114   58    0   42    0]


Compared to the Custom CNN trained on RGB and thermal images, the balanced accuracy scores only slightly increased, the accuracies even decreased. Unfortunately, training two independent CNNs for RGB and thermal images did not improve the model performance much. 

# Conclusion

All models (except the custom split) struggled to predict class 2 (Fallow Deer) and class 4 (Hybrid Pig) in the validation and test dataset correctly. Most models didn't classify a single sample of class 2 correctly. This is more understandable in the test dataset as it only includes 12 samples of this class, but in the validation dataset there are 214 images of Fallow Deer. This suggests that the Fallow Deer and Hybrid Pig samples in the validation dataset differ substantially from those in the training dataset.

Summary Table of Model Performances:<br>
*the values have been rounded to 2 decimals and sorted by balanced accuracy on the test dataset*

| Model Description                  | Train Acc. | Val Acc. | Val Balanced Acc. | Test Acc. | Test Balanced Acc. |
|------------------------------------|-----------:|---------:|------------------:|----------:|-------------------:|
| Custom Split                       | 0.77 | 0.76 | 0.71 | 0.76 | **0.72** |
| Binary Classification (RGB only)  | 0.79 | 0.75 | 0.84 | 0.68 | **0.70** |
| Binary Classification (thermal only) | 0.81 | 0.85 | 0.78 | 0.67 | **0.66** |
| Binary Classification (thermal + RGB) | 0.88 | 0.87 | 0.81 | 0.58 | **0.56** |
| Feature-Level Fusion              | 0.79 | 0.61 | 0.51 | 0.52 | **0.45** |
| Custom CNN (thermal + RGB)        | 0.88 | 0.63 | 0.50 | 0.62 | **0.41** |
| Custom CNN (thermal only)         | 0.77 | 0.47 | 0.31 | 0.43 | **0.28** |
| Transfer Learning (EfficientNet)  | 0.43 | 0.14 | 0.20 | 0.42 | **0.20** |


I also experimented with a Vision Transformer. However, it produced worse results than the other approaches and was therefore not investigated further. 

In this project, we investigated whether it is possible to classify animals belonging to five different species using only a subset of the BAMBI dataset. Unfortunately, based on the data used in this project, the answer to this question is negative. The individual flights appear to differ substantially from each other. A different distribution of flights across the training, validation, and test sets may have resulted in higher accuracies. Furthermore, reducing the number of target species would likely improve classification performance.Another issue is that the thermal and RGB images are not perfectly aligned. In some cases, the animal appears at noticeably different locations in the RGB and thermal images. This misalignment may explain why combining both modalities does not immensely improve the classification performance.